# Spinal classification — Stage 3D: CLAHE tile tuning

**Goal:** keep the CNN fixed and compare CLAHE tile grids at `clipLimit = 3.0` on the **same frozen Stage 2 dataset**.

| Trial | CLAHE clip limit | Tile grid | Output |
|---|---:|---:|---|
| C08 (existing) | 3.0 | 8×8 | Reuse Stage 3C logs and minimum-loss checkpoint. Its historical peak was 50/52 (96.15%) at epochs 10/13; its *saved minimum-loss* model scored 49/52. |
| **C10** (new) | 3.0 | 4×4 | Train and save both checkpoint types. |
| **C11** (new) | 3.0 | 16×16 | Train and save both checkpoint types. |

**Checkpoint policy:**
- `best_loss_model.keras`: minimum validation cross-entropy loss, with Keras checkpointing.
- `best_accuracy_model.keras`: maximum validation accuracy, with *lower validation loss breaking exact accuracy ties*.
- Each model is evaluated on **validation only** and saved with separate metrics and confusion matrix.

**Do not** rerun completed E0/E1/E1b/E0-G1/E0-C1/C02/C08 experiments, regenerate the split, or load any of the **51 final test images** here. Only use the dataset if you have permission to process the X-rays. Keep raw images, private manifests, and patient-identifiable information out of public GitHub. This is research, not clinical diagnosis.

**Running:** Colab → Runtime → Change runtime type → T4 GPU when available → Run all. Running again reuses complete trials and refuses to overwrite incomplete or incompatible runs.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports and private experiment paths

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_ROOT = Path('/content/drive/MyDrive/224')
SPLIT_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/splits/v1')
EXP_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/experiments/stage3')
EXP_DIR.mkdir(parents=True, exist_ok=True)

LABEL_TO_NAME = {0: 'Normal', 1: 'Scoliosis', 2: 'Spondylolisthesis'}
CLASS_TO_LABEL = {'Normal': 0, 'Scol': 1, 'Spond': 2}

GRID_DIR = EXP_DIR / 'clahe_grid_v1'  # Earlier C08 lives here.
TILE_DIR = EXP_DIR / 'clahe_tile_v1'  # New output folder for C10/C11.
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('New experiment folder:', TILE_DIR)
if not tf.config.list_physical_devices('GPU'):
    print('CPU runtime: this works but will be slower. Consider a T4 GPU.')


## 3. Verify frozen split and fingerprint

Only the training and validation partitions are loaded. The test partition is checked by metadata count, not decoded.

In [ ]:
manifest_path = SPLIT_DIR / 'split_manifest_PRIVATE.csv'
config_path = SPLIT_DIR / 'split_config_PRIVATE.json'

assert manifest_path.is_file() and config_path.is_file(), (
    'Frozen Stage 2 split not found. Run Stage 2 first; do not generate a new split.'
)
manifest = pd.read_csv(manifest_path, keep_default_na=False)
freeze = json.loads(config_path.read_text(encoding='utf-8'))

required = {'relative_path', 'class_name', 'label', 'file_sha256', 'group_id', 'split'}
assert required <= set(manifest.columns), f'Missing manifest columns: {required - set(manifest.columns)}'
assert len(manifest) == 338 and manifest['relative_path'].is_unique
assert freeze['class_to_label'] == CLASS_TO_LABEL
assert freeze['seed'] == SEED
assert freeze['patient_independent'] is False
assert set(manifest['split']) == {'train', 'validation', 'test'}
assert manifest.groupby('group_id')['split'].nunique().max() == 1, 'Grouped images cross splits.'
assert all(int(row.label) == CLASS_TO_LABEL[row.class_name] for row in manifest.itertuples(index=False))

rows = (manifest[['relative_path', 'class_name', 'label', 'file_sha256']]
        .sort_values('relative_path')
        .to_dict(orient='records'))
fingerprint = hashlib.sha256(
    json.dumps(rows, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
assert fingerprint == freeze['dataset_fingerprint'], 'Frozen dataset fingerprint mismatch.'

counts = pd.crosstab(manifest['class_name'], manifest['split']).reindex(
    index=list(CLASS_TO_LABEL), columns=['train', 'validation', 'test'], fill_value=0
)
display(counts.assign(total=counts.sum(axis=1)))
assert counts['train'].sum() == 235 and counts['validation'].sum() == 52 and counts['test'].sum() == 51

train_df = manifest[manifest['split'] == 'train'].sort_values('relative_path').reset_index(drop=True)
val_df = manifest[manifest['split'] == 'validation'].sort_values('relative_path').reset_index(drop=True)
# Intentional: no test_df or test image loader in this notebook.
assert set(train_df['relative_path']).isdisjoint(val_df['relative_path'])
print('Freeze verified. Development images:', len(train_df) + len(val_df))
print('Test set RESERVEd for final evaluation:', int(counts['test'].sum()))

## 4. Reuse exact Stage 3C preprocessing

All CLAHE transformations are per-image; no preprocessing statistics are estimated from validation/test images.

In [ ]:
from PIL import Image, ImageOps

# OpenCV CLAHE is a per-image deterministic transform; it does not fit dataset statistics.
def preprocess_clahe(path, clip_limit, tile_grid, size=128):
    if not (0.0 < clip_limit <= 10.0):
        raise ValueError('clip_limit must be positive and at most 10 for this notebook')
    if not isinstance(tile_grid, tuple) or len(tile_grid) != 2:
        raise ValueError('tile_grid must be a (rows, cols) tuple')

    with Image.open(path) as opened:
        original = ImageOps.exif_transpose(opened).convert('RGB')
        resized = ImageOps.pad(
            original, (size, size), method=Image.Resampling.LANCZOS,
            color=(0, 0, 0),
        )
        rgb = np.asarray(resized, dtype=np.uint8)

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    enhanced = cv2.createCLAHE(
        clipLimit=float(clip_limit), tileGridSize=tuple(tile_grid)
    ).apply(gray)
    return enhanced[..., np.newaxis].astype(np.float32) / 255.0


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def load_tuned_partition(rows, clip_limit, tile_grid):
    X, y = [], []
    for item in rows.itertuples(index=False):
        path = DATA_ROOT / item.relative_path
        if not path.is_file() or file_sha256(path) != item.file_sha256:
            raise RuntimeError('An image changed since the frozen split; stop and review.')
        X.append(preprocess_clahe(path, clip_limit, tile_grid))
        y.append(int(item.label))
    return np.stack(X), np.asarray(y, dtype=np.int64)

# Sanity check on training image only.
example_path = DATA_ROOT / train_df.iloc[0]['relative_path']
for clip in (1.0, 2.0, 3.0):
    arr = preprocess_clahe(example_path, clip, (8, 8))
    assert arr.shape == (128, 128, 1) and arr.dtype == np.float32
    assert np.isfinite(arr).all() and 0 <= arr.min() <= arr.max() <= 1
print('CLAHE image preprocessing sanity checks passed.')


## 5. Reference CNN unchanged

Same architecture, optimizer, batch size and class weighting as the E0-C1 and C08 runs.

In [ ]:
from tensorflow.keras import layers

def make_reference_cnn(shape):
    return tf.keras.Sequential([
        layers.Input(shape=shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(3, activation='softmax'),
    ], name='reference_cnn')

def make_model(shape):
    model = make_reference_cnn(shape)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    return model

# Inspect architecture using the correct one-channel input.
preview = make_model((128, 128, 1))
print('Reference CNN parameters (grayscale):', preview.count_params())
del preview
tf.keras.backend.clear_session()


## 6. Register tile experiments and verify existing comparison baselines

C10 and C11 use unique IDs and a **new output directory**, so older CLAHE grid files remain intact.

In [ ]:
TRIALS = {
    'C10': {'clip_limit': 3.0, 'tile_grid': (4, 4)},
    'C11': {'clip_limit': 3.0, 'tile_grid': (16, 16)},
}
RUN_IDS = ['C10', 'C11']   # Run both; to run one, change to ['C10'] or ['C11'].
EPOCHS = 40
BATCH_SIZE = 16
assert RUN_IDS and len(set(RUN_IDS)) == len(RUN_IDS)
assert all(trial in TRIALS for trial in RUN_IDS)

# Verify existing reference scores without opening test images.
BASELINES = {
    'E0-G1': EXP_DIR / 'E0-G1',
    'E0-C1': EXP_DIR / 'E0-C1',
    'C08': GRID_DIR / 'C08',
}
for name, directory in BASELINES.items():
    p = directory / 'validation_metrics.json'
    cfg = directory / 'configuration.json'
    assert p.is_file() and cfg.is_file(), f'Missing saved baseline {name}: {directory}'
    saved_cfg = json.loads(cfg.read_text(encoding='utf-8'))
    saved = json.loads(p.read_text(encoding='utf-8'))
    assert saved_cfg['dataset_fingerprint'] == fingerprint, f'Different dataset for {name}'
    assert saved.get('test_evaluated') is False, f'Test was evaluated for {name}'
    print(f'{name}: saved validation accuracy {saved["val_accuracy"]:.2%} '
          f'and macro F1 {saved["val_macro_f1"]:.4f}')

# Sanity check on training image: tile size actually changes preprocessed pixels.
example_path = DATA_ROOT / train_df.iloc[0]['relative_path']
images = {
    tiles: preprocess_clahe(example_path, 3.0, (tiles, tiles))
    for tiles in (4, 8, 16)
}
for tiles in images:
    assert images[tiles].shape == (128, 128, 1)
    assert np.isfinite(images[tiles]).all()
for a, b in ((4, 8), (8, 16)):
    mean_delta = float(np.abs(images[a] - images[b]).mean())
    print(f'Sample mean absolute pixel change {a}x{a} versus {b}x{b}: {mean_delta:.6f}')
    if mean_delta == 0:
        print('WARNING: sampled image preprocessing is identical; evaluate usefulness of this trial.')

print('New trials:', RUN_IDS)
for trial in RUN_IDS:
    folder = TILE_DIR / trial
    print(trial, ': existing directory (verify/skip)' if folder.exists() else ': ready')


## 7. Save maximum-accuracy checkpoint with validation-loss tie-break

Only a true improvement in accuracy, or a lower loss at exactly tied accuracy, replaces the run-local accuracy checkpoint.

In [ ]:
class SaveBestAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, path):
        super().__init__()
        self.path = str(path)
        self.best_accuracy = float('-inf')
        self.best_loss = float('inf')
        self.best_epoch = None

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        accuracy = float(logs['val_accuracy'])
        loss = float(logs['val_loss'])
        better_accuracy = accuracy > self.best_accuracy + 1e-8
        equal_accuracy_better_loss = (
            abs(accuracy - self.best_accuracy) <= 1e-8
            and loss < self.best_loss - 1e-8
        )
        if better_accuracy or equal_accuracy_better_loss:
            self.best_accuracy, self.best_loss = accuracy, loss
            self.best_epoch = epoch + 1
            self.model.save(self.path)


def validation_summary(model_path, X_val, y_val):
    fitted = tf.keras.models.load_model(str(model_path))
    scores = fitted.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
    predicted = np.argmax(scores, axis=1)
    cm = confusion_matrix(y_val, predicted, labels=[0, 1, 2])
    report = classification_report(
        y_val, predicted, labels=[0, 1, 2],
        target_names=[LABEL_TO_NAME[i] for i in range(3)],
        output_dict=True, zero_division=0,
    )
    return {
        'accuracy': float(accuracy_score(y_val, predicted)),
        'macro_f1': float(f1_score(
            y_val, predicted, labels=[0, 1, 2],
            average='macro', zero_division=0
        )),
        'balanced_accuracy': float(balanced_accuracy_score(y_val, predicted)),
        'per_class_recall': {
            name: float(report[name]['recall'])
            for name in LABEL_TO_NAME.values()
        },
        'confusion_matrix': cm.tolist(),
    }


## 8. Train each trial (append-only)

If a completed trial exists with matching settings, reuse it. If an incomplete or incompatible trial folder exists, stop instead of overwriting.

In [ ]:
def run_trial(trial_id):
    if trial_id not in TRIALS:
        raise KeyError(f'Unknown trial: {trial_id}')
    settings = TRIALS[trial_id]
    clip = settings['clip_limit']
    tiles = settings['tile_grid']
    folder = TILE_DIR / trial_id
    cfg_path = folder / 'configuration.json'
    metric_path = folder / 'validation_metrics.json'
    loss_path = folder / 'best_loss_model.keras'
    accuracy_path = folder / 'best_accuracy_model.keras'

    # Re-running the notebook never erases a previous experiment.
    if folder.exists():
        needed = [cfg_path, metric_path, loss_path, accuracy_path]
        if not all(path.is_file() for path in needed):
            raise RuntimeError(
                f'Incomplete {trial_id} folder exists: {folder}. '
                'Please inspect it manually; do not overwrite automatically.'
            )
        saved_cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
        expected = {
            'trial': trial_id, 'clip_limit': clip,
            'tile_grid': list(tiles), 'dataset_fingerprint': fingerprint,
            'split_version': 'v1', 'seed': SEED,
            'epochs_max': EPOCHS, 'batch_size': BATCH_SIZE,
            'checkpoint_policy': 'best_loss_and_best_accuracy_loss_tiebreak',
        }
        for key, value in expected.items():
            if saved_cfg.get(key) != value:
                raise RuntimeError(f'{trial_id} saved run differs on {key}; stop.')
        metrics = json.loads(metric_path.read_text(encoding='utf-8'))
        assert metrics.get('test_evaluated') is False
        print(f'{trial_id}: already complete; reusing saved validation metrics.')
        return metrics

    X_train, y_train = load_tuned_partition(train_df, clip, tiles)
    X_val, y_val = load_tuned_partition(val_df, clip, tiles)
    assert X_train.shape == (235, 128, 128, 1)
    assert X_val.shape == (52, 128, 128, 1)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = make_model(X_train.shape[1:])
    weights = compute_class_weight('balanced', classes=np.arange(3), y=y_train)
    class_weights = {int(i): float(w) for i, w in enumerate(weights)}

    config = {
        'trial': trial_id, 'architecture': 'reference',
        'clip_limit': clip, 'tile_grid': list(tiles),
        'image_size': [128, 128], 'mode': 'grayscale_clahe',
        'normalization': 'uint8 CLAHE -> float32 / 255',
        'optimizer': 'Adam', 'initial_learning_rate': 3e-4,
        'checkpoint_policy': 'best_loss_and_best_accuracy_loss_tiebreak',
        'early_stopping': 'val_loss patience=9',
        'learning_rate_schedule': 'val_loss plateau factor=0.5 patience=4',
        'dataset_fingerprint': fingerprint, 'split_version': 'v1',
        'seed': SEED, 'epochs_max': EPOCHS, 'batch_size': BATCH_SIZE,
        'class_weights': class_weights, 'tensorflow_version': str(tf.__version__),
        'opencv_version': str(cv2.__version__),
        'train_images': len(y_train), 'validation_images': len(y_val),
        'patient_independent': False, 'test_evaluated': False,
        'model_parameters': int(model.count_params()),
    }

    folder.mkdir(parents=True, exist_ok=False)
    cfg_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
    accuracy_callback = SaveBestAccuracy(accuracy_path)
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(loss_path), monitor='val_loss', mode='min', save_best_only=True
        ),
        accuracy_callback,
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=9, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(folder / 'epoch_log.csv')),
    ]
    print(f'\n=== {trial_id}: clipLimit={clip}, tileGrid={tiles} ===')
    began = time.perf_counter()
    history = model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        class_weight=class_weights, callbacks=callbacks,
        shuffle=True, verbose=2,
    )
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(folder / 'history.csv', index=False)

    # Two separate checkpoint policies: report their actual saved behavior.
    loss_metrics = validation_summary(loss_path, X_val, y_val)
    accuracy_metrics = validation_summary(accuracy_path, X_val, y_val)
    loss_epoch = int(np.argmin(history.history['val_loss']) + 1)
    assert abs(accuracy_metrics['accuracy'] - accuracy_callback.best_accuracy) < 1e-5

    result = {
        'trial': trial_id, 'clip_limit': clip, 'tile_grid': f'{tiles[0]}x{tiles[1]}',
        'min_val_loss': float(min(history.history['val_loss'])),
        'best_loss_epoch': loss_epoch,
        'best_accuracy_epoch': accuracy_callback.best_epoch,
        'best_accuracy_epoch_val_loss': accuracy_callback.best_loss,
        'best_accuracy_epoch_val_accuracy': accuracy_callback.best_accuracy,
        'peak_accuracy_observed': float(max(history.history['val_accuracy'])),
        'loss_checkpoint': loss_metrics,
        'accuracy_checkpoint': accuracy_metrics,
        'epochs_completed': len(history.history['loss']),
        'runtime_seconds': time.perf_counter() - began,
        'test_evaluated': False,
    }
    for key, cm in [('loss', loss_metrics['confusion_matrix']),
                    ('accuracy', accuracy_metrics['confusion_matrix'])]:
        np.save(folder / f'confusion_matrix_{key}_validation.npy', np.asarray(cm))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for metric, ax in (('loss', axes[0]), ('accuracy', axes[1])):
        ax.plot(history.history[metric], label='training')
        ax.plot(history.history[f'val_{metric}'], label='validation')
        ax.set_title(f'{trial_id} — {metric}')
        ax.set_xlabel('Epoch')
        ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'learning_curves.png', dpi=150)
    plt.show()
    plt.close(fig)

    print('Best-loss model:', loss_epoch, loss_metrics)
    print('Best-accuracy model:', accuracy_callback.best_epoch, accuracy_metrics)
    # Save results LAST so an incomplete run is never interpreted as finished.
    metric_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
    return result

results = [run_trial(trial) for trial in RUN_IDS]
print('All requested trials complete (or verified and reused).')


## 9. Compare baseline and new runs without accessing held-out test images

**C08’s 96.15% was a peak epoch, but its saved best-loss checkpoint had 94.23% validation accuracy.** Old runs were saved under a single checkpoint rule; new runs save both types.

In [ ]:
rows = []
for exp_id, directory in BASELINES.items():
    metrics = json.loads((directory / 'validation_metrics.json').read_text('utf-8'))
    assert metrics['test_evaluated'] is False
    item = {
        'trial': exp_id, 'checkpoint': 'best loss (historical)',
        'val_accuracy': float(metrics['val_accuracy']),
        'val_macro_f1': float(metrics['val_macro_f1']),
        'val_balanced_accuracy': float(metrics['val_balanced_accuracy']),
        'best_val_loss': float(metrics['best_val_loss']),
    }
    rows.append(item)

for exp_id in TRIALS:
    path = TILE_DIR / exp_id / 'validation_metrics.json'
    if not path.is_file():
        continue
    saved = json.loads(path.read_text(encoding='utf-8'))
    assert saved['test_evaluated'] is False
    for checkpoint, data in [
        ('best loss', saved['loss_checkpoint']),
        ('best accuracy', saved['accuracy_checkpoint']),
    ]:
        rows.append({
            'trial': exp_id, 'checkpoint': checkpoint,
            'val_accuracy': data['accuracy'],
            'val_macro_f1': data['macro_f1'],
            'val_balanced_accuracy': data['balanced_accuracy'],
            'best_val_loss': saved['min_val_loss'] if checkpoint == 'best loss' else saved['best_accuracy_epoch_val_loss'],
        })
comparison = pd.DataFrame(rows).sort_values(
    ['val_macro_f1', 'best_val_loss'], ascending=[False, True]
)
display(comparison.round(4).reset_index(drop=True))
print('Test set: NOT evaluated. Grouping by patient remains unverified.')


## Stage 3D checkpoint and what comes afterward

- Evaluate validation accuracy, **macro F1**, per-class recall, and loss for each checkpoint; a 1-image difference on 52 validation images is only **1.92 percentage points**.
- Verify C10/C11 improved *beyond just a single favorable epoch* and inspect the saved loss-vs-accuracy checkpoint tradeoff.
- Only after these runs, repeat two promising configurations over **precommitted independent random seeds**. Use a new experiment folder per seed and a clear aggregate comparison, still without touching the test set.
- Because repeated comparison against 52 validation images creates selection bias, don't overstate a 96% peak as robust evidence of generalization. A truly patient-independent dataset would be needed for medical claims.
- Keep the frozen split unchanged and reserve all 51 test images for one final evaluation after selecting the training protocol.
